# Nassau Candy Distributor - Exploratory Data Analysis & Route Clustering

This notebook provides empirical evidence, distribution analyses, and algorithmic clustering for the **Factory Reallocation and Shipping Optimization System**.

### Core Objectives:
1. Audit raw shipment transactions and validate financial accounting consistency.
2. Characterize derived lead time distributions, handling the multi-year date shift anomaly.
3. Compute spatial haversine distances bridging manufacturing facilities to destination states.
4. Execute KMeans route clustering with silhouette validation to identify slow and congested corridors.
5. Evaluate predictive models (Linear Regression, Random Forest, Gradient Boosting) for like-for-like scenario simulation.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_PATH, FACTORIES, PRODUCT_FACTORY_MAP
from src.data_prep import run_stage_1
from src.features import engineer_features
from src.clustering import run_stage_3
from src.models import run_stage_4

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Data Cleaning and Financial Validation

In [ ]:
df_clean, stage1_summary = run_stage_1(save_output=False)
print("Stage 1 Summary:")
print("Raw rows:", stage1_summary["raw_rows"])
print("Cleaned rows retained:", stage1_summary["cleaned_rows"])
print("Financial Discrepancy Count:", stage1_summary["validation"]["discrepant_rows_count"])
print("Outlier audit:", stage1_summary["outlier_audit"])


## 2. Lead Time Days Distribution & Limitation Audit

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_clean["lead_time_days"], kde=True, ax=ax[0], color="#2563eb", bins=30)
ax[0].set_title("Distribution of Derived Lead Time Index (Days)")
ax[0].set_xlabel("Lead Time Days")

sns.boxplot(x="Division", y="lead_time_days", data=df_clean, ax=ax[1], palette="Blues")
ax[1].set_title("Lead Time by Product Division")
plt.tight_layout()
plt.show()

print(df_clean["lead_time_days"].describe())


## 3. Ship Mode Analysis (Statistical Invariance)

In [ ]:
ship_mode_stats = df_clean.groupby("Ship Mode")["lead_time_days"].agg(["count", "mean", "std", "median"])
print(ship_mode_stats)

plt.figure(figsize=(8, 4))
sns.barplot(x=ship_mode_stats.index, y=ship_mode_stats["mean"], palette="crest")
plt.title("Mean Lead Time by Ship Mode")
plt.ylabel("Mean Lead Time Days")
plt.ylim(1200, 1400)
plt.show()


## 4. Feature Engineering: Spatial Distance Bridge

In [ ]:
df_feat, feat_audit = engineer_features(df_clean)
print("Feature Engineering Audit:")
print(f"Mean Haversine Distance: {feat_audit['mean_distance_km']:.2f} km")
print(f"Unmatched States: {feat_audit['unmatched_states']}")
print(f"Unique Logistical Corridors: {feat_audit['unique_routes']}")


## 5. KMeans Route Clustering and Silhouette Validation

In [ ]:
clustered_routes, eda_dists, cluster_meta = run_stage_3(save_model=False)

print(f"Optimal Clusters: k={cluster_meta['best_k']} (Silhouette Score: {cluster_meta['best_silhouette']:.4f})")
print("\nSilhouette Scores by Candidate k:")
for k, s in cluster_meta["silhouette_scores"].items():
    print(f"  k={k}: {s:.4f}")

print("\nCluster Profiles:")
print(cluster_meta["cluster_profiles"][["cluster_id", "cluster_label", "route_count", "avg_lead_time_days", "avg_order_volume"]])


## 6. Predictive Modeling Benchmarking (5-Fold CV)

In [ ]:
metrics_df, feat_imp_df, chosen_model, summary = run_stage_4(save_artifacts=False)
print("Model Comparison Table:")
print(metrics_df.to_string(index=False))

print(f"\nSelected Model: {chosen_model}")

plt.figure(figsize=(10, 5))
sns.barplot(x="Importance", y="Feature", data=feat_imp_df.head(8), palette="viridis")
plt.title("Top Predictive Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()
